In [ ]:
############## Code to split pdf to keep only pages with Rating table in it #####################
import pdfplumber
from PyPDF2 import PdfWriter, PdfReader

input_pdf_path = "Axis Regular Fund Factsheet June-2025.pdf"
output_pdf_path = "filtered{}.pdf".format(input_pdf_path.split(".")[0])

reader = PdfReader(input_pdf_path)
writer = PdfWriter()

with pdfplumber.open(input_pdf_path) as pdf:
    for page_num, page in enumerate(pdf.pages):
        words = page.extract_words()
        capital_words = [word['text'] for word in words if word['text'].isupper()]
        if "PORTFOLIO" in capital_words:
            for word in words:
                if 'Rating' in word['text']:
                    print(f"\n=== Page {page_num + 1} ===")
                    # print(page_num,capital_words)
                    print([word['text'] for word in words])
                    writer.add_page(reader.pages[page_num])
                    break  # add this page once, then move to next page

# Save the filtered PDF
with open(output_pdf_path, "wb") as f_out:
    writer.write(f_out)




=== Page 60 ===
['Tata', 'Aggressive', 'Hybrid', 'Fund', '(An', 'open', 'ended', 'hybrid', 'scheme', 'investing', 'predominantly', 'in', 'equity', '&', 'equity', 'related', 'instruments.)', 'As', 'on', '30th', 'June', '2025', 'PORTFOLIO', 'No.', 'of', 'Market', 'Value', '%', 'of', 'Market', 'Value', '%', 'to', 'INVESTMENT', 'STYLE', 'Company', 'Name', 'Name', 'of', 'the', 'Instrument', 'Ratings', 'Shares', 'Rs.', 'Lakhs', 'Assets', 'Rs.', 'Lakhs', 'NAV', 'Invests', '65%', 'to', '80%', 'investment', 'in', 'Equity', '&', 'equity', 'related', 'i', 'i', 'e', 's', 'n', 'n', 'u', 'q', 's', 's', 'b', 'u', 't', 't', 'r', 'r', 'j', 'i', 'e', 't', 'u', 'u', 'y', 'c', 'm', 'm', 't', 's', 't', 'e', 'e', 'c', 'o', 'n', 'n', 'h', 'a', 't', 't', 'e', 's', 's', 'v', 'm', '.', 'a', '&', 'e', 'F', 'il', 'o', '.', 'a', '2', 'b', 'r', '(M', '0', 'il', 't', '%', 'i', 'a', 'o', 'ty', 'x', 'n', 'a', 't', 'o', 't', 'o', 'h', 't', 'f', 'i', 'l', 'o', 'd', 'y', '3', 'n', 'i', '5', 's', 'I', 'n', 't', '%', 'p',

In [2]:
############ Code to save screenshot of right_half of every page that contains rating table ################
import os
from pdf2image import convert_from_path
import cv2
import numpy as np

# === Configuration ===
pdf_path = "filteredAxis Regular Fund Factsheet June-2025.pdf"  # Replace with your actual PDF file path
output_dir = "right_half_screenshots"
os.makedirs(output_dir, exist_ok=True)

# === Step 1: Convert PDF pages to images ===
images = convert_from_path(pdf_path, dpi=300)  # Use high DPI for better clarity

# === Step 2: Crop right half and save ===
for idx, pil_image in enumerate(images):
    # Convert PIL to OpenCV (BGR)
    image = np.array(pil_image)
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    # Get dimensions
    height, width, _ = image.shape

    # Crop right half (60% to 100%)
    x_start = int(0.45 * width)
    y_start = int(0.15 * height)  # 15% from top
    right_half = image[y_start:, x_start:]
    # right_half = image[:, x_start:]

    # Save cropped image
    output_path = os.path.join(output_dir, f"page_{idx+1}_right_half.png")
    cv2.imwrite(output_path, right_half)

    print(f"✅ Saved: {output_path}")

print("\n✅ All screenshots of right half completed.")


✅ Saved: right_half_screenshots\page_1_right_half.png
✅ Saved: right_half_screenshots\page_2_right_half.png
✅ Saved: right_half_screenshots\page_3_right_half.png
✅ Saved: right_half_screenshots\page_4_right_half.png
✅ Saved: right_half_screenshots\page_5_right_half.png
✅ Saved: right_half_screenshots\page_6_right_half.png
✅ Saved: right_half_screenshots\page_7_right_half.png
✅ Saved: right_half_screenshots\page_8_right_half.png
✅ Saved: right_half_screenshots\page_9_right_half.png
✅ Saved: right_half_screenshots\page_10_right_half.png
✅ Saved: right_half_screenshots\page_11_right_half.png
✅ Saved: right_half_screenshots\page_12_right_half.png
✅ Saved: right_half_screenshots\page_13_right_half.png
✅ Saved: right_half_screenshots\page_14_right_half.png
✅ Saved: right_half_screenshots\page_15_right_half.png
✅ Saved: right_half_screenshots\page_16_right_half.png
✅ Saved: right_half_screenshots\page_17_right_half.png
✅ Saved: right_half_screenshots\page_18_right_half.png
✅ Saved: right_half

In [1]:
import cv2
import numpy as np
import pytesseract
from PIL import Image
import os
import re

rating_set = set()
# Set Tesseract path (update for your system)
pytesseract.pytesseract.tesseract_cmd = r'C:\\Program Files\\Tesseract-OCR\\tesseract.exe'
pan = 'AACTA5925A'
mutual_fund = "AXIS MUTUAL FUND"
pattern = re.compile(r"(AAA\(SO\)|AAA\(CE\)|AA\(CE\)|A1\+|AA\+|AA-|AAA|AA|SOVEREIGN|SOVRN SOV|SOVRN|SOV|Sovereign)")
agencies = ["CARE", "IND", "CRISIL", "ICRA", "FITCH", "India Ratings", "Brickwork", "Acuite"]


agency_pattern = re.compile(rf"({'|'.join(agencies)})")


def normalize_rating_text(text):
    replacements = {
        "＋": "+",  # fullwidth plus
        "–": "-",  # en dash
        "−": "-",  # minus
        "﹣": "-",  # small hyphen
        "‒": "-",  # figure dash
        "―": "-",  # horizontal bar
        "（": "(",  # fullwidth left parenthesis
        "）": ")",  # fullwidth right parenthesis
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)

    # Remove zero-width and non-breaking spaces
    text = re.sub(r"[\u200B-\u200D\uFEFF\u00A0]", "", text)
    return text

def agency_matching(normalized_text, agency=''):
    agency_match = agency_pattern.search(normalized_text.replace("INDIA", ""))
    if agency_match:
        agency = agency_match.group(1)
        print("agency:", agency)
        normalized_text = normalized_text.replace(agency, " ").replace("( )", " ")
        print("agency_replaced_normalised_text :", normalized_text)
    return [normalized_text, agency]

def rating_matching(normalized_text, rating=''):
    match = pattern.search(normalized_text)
    # print(f"Line: {normalized_text}")
    if match:
        rating = match.group(0)
        print(f"Rating: {rating}\n")
        normalized_text = normalized_text.replace(rating, " ")
        print("rating_replaced_normalised_text :", normalized_text)
    return [normalized_text, rating]

def AUM_matching(normalized_text, total_AUM=''):
    match_AUM = re.findall(r"\d+\.\d+", normalized_text)

    # If found, return the last one (but not if it's at start)
    if match_AUM:
        last = match_AUM[-1]
        # check position of last match
        m = re.finditer(r"\d+\.\d+", normalized_text)
        last_match = list(m)[-1]
        if last_match.start() == 0:
            total_AUM = None  # ignore if at start
        else:
            total_AUM = last
            # total_AUM = match_AUM[-1]
            print(f"Total AUM: {total_AUM}\n")
            normalized_text = normalized_text.replace(total_AUM, "")
            print("AUM_replaced_normalised_text :", normalized_text)
    return [normalized_text, total_AUM]

In [5]:
import cv2
import pytesseract
from PIL import Image
import os
import pandas as pd

# OPTIONAL (Windows only): specify path to tesseract.exe
pytesseract.pytesseract.tesseract_cmd = r'C:\\Program Files\\Tesseract-OCR\\tesseract.exe'
input_dir = "right_half_screenshots"
# output_dir = "final_images"
# os.makedirs(output_dir, exist_ok=True)
correct_rating_df = pd.DataFrame(columns=['Pan', 'ISIN No.', 'Mutual Fund', 'Fund Name','Fund Type', 'Portfolio', "Agency", 'Ratings', "Category", '(%) Of Total AUM', "Factsheet Date", "URL", 'Filename'])

for filename in os.listdir(input_dir):
    print("---------------------------------------------------------->>>>>>>>>>>>>>>>>>>>>>>>--------------------------------------------------------------")
    if filename.endswith(".png"):
        start_printing = False
        image_path = os.path.join(input_dir, filename)
        print(image_path)
        image = cv2.imread(image_path)
        # Load image using OpenCV
        # image = cv2.imread(r"D:\Nexensus_Projects\pdfEtraction\LIC\right_half_screenshots_bandhan\page_2_right_half.png")

        # Convert to grayscale for better OCR

        # scale_percent = 200
        # width = int(image.shape[1] * scale_percent / 100)
        # height = int(image.shape[0] * scale_percent / 100)
        # dim = (width, height)

        # # Resize image
        # image = cv2.resize(image, dim, interpolation=cv2.INTER_LINEAR)
        # gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # # (Optional) Apply thresholding for better results
        # # gray = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]

        # # Save preprocessed image temporarily (optional)
        # cv2.imwrite("temp_gray.png", gray)

        # Extract text
        text = pytesseract.image_to_string(cv2.imread(image_path))
        # # print(text)
        # if "name of the instruments rating % to nav" not in text.lower():
        #     print("not here")
        #     text = pytesseract.image_to_string(cv2.imread(image_path))
        for line in text.split("\n"):
            # print(line)
            # line = line.strip()
            if line:
                print(line)
                print("<><><<><>>>>>>>>>>>>>>><<><><><>><<<<<<<<<<<<<<<<><><><><<><<<>>>>><><><>")
                agency = ""
                rating = ""
                portfolio = ""
                total_AUM = ''
                # if "PORTFOLIO" in line or "Rating" in line or line.startswith("There is") or line is None:
                #     continue
                # elif line[0].islower() and len(line) > 60:
                #     # print("skipping ",line )
                # #     continue
                # if "Name of the Instruments"  in line:
                #     start_printing = True
                #     continue
                # if  line.strip() == "—_" or line.strip() == "°" or line.strip() == "is)" or re.fullmatch(r"\d+", line.strip()):
                #     continue
                # # if "Grand Total" in line and start_printing:
                # if "Other Current Assets" in line:
                #     print(line)
                #     normalized_text = normalize_rating_text(line)

                #     AUM_matching_output = AUM_matching(normalized_text)
                #     normalized_text = AUM_matching_output[0]
                #     total_AUM = AUM_matching_output[1]

                #     temp_df = pd.DataFrame([{
                #     "Pan" : pan,
                #     "ISIN No." : None,
                #     "Mutual Fund" : mutual_fund,
                #     "Fund Name" : "CANARA ROBECO " + filename.split("_page")[0].replace("_", " "),
                #     "Fund Type" : filename.split("_page")[0].replace("_", " "),
                #     "Portfolio"  : normalized_text,
                #     "Agency" : None,
                #     "Ratings" : None,
                #     "Category" : None,
                #     "(%) Of Total AUM" : total_AUM,
                #     "Factsheet Date" : None,
                #     "URL" : None,
                #     "Filename" : filename,
                #     }])
                #     correct_rating_df = pd.concat([correct_rating_df, temp_df])

                #     ############# Manually adding Grand total row ##############################
                #     temp_df = pd.DataFrame([{
                #     "Pan" : pan,
                #     "ISIN No." : None,
                #     "Mutual Fund" : mutual_fund,
                #     "Fund Name" : "CANARA ROBECO " + filename.split("_page")[0].replace("_", " "),
                #     "Fund Type" : filename.split("_page")[0].replace("_", " "),
                #     "Portfolio"  : "Grand Total ( Net Asset)",
                #     "Agency" : None,
                #     "Ratings" : None,
                #     "Category" : None,
                #     "(%) Of Total AUM" : "100.00",
                #     "Factsheet Date" : None,
                #     "URL" : None,
                #     "Filename" : filename,
                #     }])
                #     correct_rating_df = pd.concat([correct_rating_df, temp_df])
                #     break
                # # elif "SCHEME" in line:
                # #     break
                # elif start_printing:
                #     print("else block", line)
                #     normalized_text = normalize_rating_text(line)

                #     agency_matching_output = agency_matching(normalized_text)
                #     normalized_text = agency_matching_output[0]
                #     agency = agency_matching_output[1]


                #     rating_matching_output = rating_matching(normalized_text)
                #     normalized_text = rating_matching_output[0]
                #     rating = rating_matching_output[1]

                #     AUM_matching_output = AUM_matching(normalized_text)
                #     normalized_text = AUM_matching_output[0]
                #     total_AUM = AUM_matching_output[1]

                #     temp_df = pd.DataFrame([{
                #     "Pan" : pan,
                #     "ISIN No." : None,
                #     "Mutual Fund" : mutual_fund,
                #     "Fund Name" : "CANARA ROBECO " + filename.split("_page")[0].replace("_", " "),
                #     "Fund Type" : filename.split("_page")[0].replace("_", " "),
                #     "Portfolio"  : normalized_text,
                #     "Agency" : agency,
                #     "Ratings" : rating,
                #     "Category" : None,
                #     "(%) Of Total AUM" : total_AUM,
                #     "Factsheet Date" : None,
                #     "URL" : None,
                #     "Filename" : filename,
                #     }])
                #     correct_rating_df = pd.concat([correct_rating_df, temp_df])

                # print(",,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,")

---------------------------------------------------------->>>>>>>>>>>>>>>>>>>>>>>>--------------------------------------------------------------
right_half_screenshots\page_10_right_half.png
Instrument Type/ Issuer Name Rating % of NAV
<><><<><>>>>>>>>>>>>>>><<><><><>><<<<<<<<<<<<<<<<><><><><<><<<>>>>><><><>
Corporate Bond 80.02%
<><><<><>>>>>>>>>>>>>>><<><><><>><<<<<<<<<<<<<<<<><><><><<><<<>>>>><><><>
Kohima-Mariani Transmission Limited IND AAA 5.38%
<><><<><>>>>>>>>>>>>>>><<><><><>><<<<<<<<<<<<<<<<><><><><<><<<>>>>><><><>
Vedanta Limited CRISIL AA/ICRA AA 4.65%
<><><<><>>>>>>>>>>>>>>><<><><><>><<<<<<<<<<<<<<<<><><><><<><<<>>>>><><><>
Birla Corporation Limited ICRA AA 443%
<><><<><>>>>>>>>>>>>>>><<><><><>><<<<<<<<<<<<<<<<><><><><<><<<>>>>><><><>
Aditya Birla Renewables Limited CRISIL AA 4.18%
<><><<><>>>>>>>>>>>>>>><<><><><>><<<<<<<<<<<<<<<<><><><><<><<<>>>>><><><>
Narayana Hrudayalaya Limited ICRA AA 4.16%
<><><<><>>>>>>>>>>>>>>><<><><><>><<<<<<<<<<<<<<<<><><><><<><<<>>>>><><><>
Nirm

In [2]:
####################### Code to fetch text from screenshot after zooom ######################
import cv2
import numpy as np
import pytesseract
from PIL import Image
import os
import re

rating_set = set()
# Set Tesseract path (update for your system)
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

input_dir = "right_half_screenshots"

import pandas as pd
correct_rating_df = pd.DataFrame(columns=['Name of the Instrument / Company Name', 'Ratings', 'Market Value Rs Lakhs', '% to NAV'])
remaining_rating_df = pd.DataFrame(columns=['Text'])
for filename in os.listdir(input_dir)[:]:
    print(">>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>")
    start_printing = False
    if filename.endswith(".png"):
        image_path = os.path.join(input_dir, filename)
        print(image_path)
        image = cv2.imread(image_path)

        # Scale image (e.g., 1.5x zoom)
        # scale_percent = 120  # Adjust this value as needed (e.g., 150 means 150% size)
        scale_percent = 200
        width = int(image.shape[1] * scale_percent / 100)
        height = int(image.shape[0] * scale_percent / 100)
        dim = (width, height)

        # Resize image
        image = cv2.resize(image, dim, interpolation=cv2.INTER_LINEAR)

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Threshold to isolate dark lines (black separators)
        # _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

        # Adaptive threshold to handle gray lines and varying lighting
        thresh = cv2.adaptiveThreshold(
            gray, 255,
            cv2.ADAPTIVE_THRESH_MEAN_C,
            cv2.THRESH_BINARY_INV,
            blockSize=15,
            C=10
        )

        # Detect horizontal lines
        horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (50, 1))
        detected_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)

        # Find contours of the lines
        contours, _ = cv2.findContours(detected_lines, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Get y-coordinates of all horizontal lines
        line_y_coords = sorted([cv2.boundingRect(cnt)[1] for cnt in contours])

        # Add image boundaries
        line_y_coords = [0] + line_y_coords + [image.shape[0]]

        # Process and print each row
        print("\nExtracted Table Rows:")
        for i in range(len(line_y_coords) - 1):
            y_start = line_y_coords[i]
            y_end = line_y_coords[i + 1]
            
            # Skip very small regions (likely just the line itself)
            if y_end - y_start < 10:
                continue

            # Extract the row region
            row_img = image[y_start:y_end, :]

            # OPTIONAL: Resize the row to improve OCR accuracy
            zoom_factor = 2.0
            row_img = cv2.resize(row_img, None, fx=zoom_factor, fy=zoom_factor, interpolation=cv2.INTER_LINEAR)

            # Convert to PIL Image for Tesseract
            row_pil = Image.fromarray(cv2.cvtColor(row_img, cv2.COLOR_BGR2RGB))

            # Tesseract OCR
            custom_config = r'--oem 3 --psm 6 -c preserve_interword_spaces=1'
            text = pytesseract.image_to_string(row_pil, config=custom_config).strip()

            if text:
                # if "PORTFOLIO" in text:
                #     print("portfoilio text :::::::::")
                #     start_printing = True
                if "Issuer" in text:
                    # print("Issuer text :::::::::", text)
                    start_printing = True
                if start_printing:
                    if "Issuer" in text:
                        headers = re.split(r'\s{2,}', text)
                        print("headers :", headers)
                        # pass
                    else:
                        lines = re.split(r'\s{2,}', text)
                        print("headers :", lines)
                        pass
                    # else:
                    #     for t in text.split("\n"):
                    #         # print(t)
                    #         print(len(re.split(r'\s{2,}', t)), re.split(r'\s{2,}', t))
                    #         print("-------------------------------------------------------------------------")
                if "Grand Total" in text and start_printing:
                    # print("Grand total text ::::::::::::::::::::", text)
                    break

>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
right_half_screenshots\page_10_right_half.png

Extracted Table Rows:
headers : ['|', 'Instrument Type/ Issuer Name', 'Rating', '% of NAV', ')']
headers : ['| Corporate Bond', '80.02% |']
headers : ['Kohima-Mariani Transmission Limited', 'IND AAA', '5.38%\nVedanta Limited', 'CRISIL AA/ICRA AA', '4.65%\nBirla Corporation Limited', 'ICRA AA', '4.43%\nAditya Birla Renewables Limited', 'CRISIL AA', '4.18%\nNarayana Hrudayalaya Limited', 'ICRA AA', '4.16%\nNirma Limited', 'CRISIL AA', '4.15%\nInfopark Properties Limited', 'CARE AA-', '4.13%\nAltius Telecom Infrastructure Trust', 'CRISIL AAA', '4.13%\nAditya Birla Real Estate Limited', 'CRISIL AA', '4.11%\nAditya Birla Digital Fashion Ventures Limited', 'CRISIL AA-', '4.10%\nDelhi International Airport Limited', 'ICRA AA', '4.09%\nNuvama Wealth Finance Limited', 'CARE AA-', '3.28%\nGMR Hyderabad International Airport Limited', 'IND AA+', '2.78%\neam Hotel 

KeyboardInterrupt: 

In [6]:
################## Testing text fetching code

import cv2
import numpy as np
import pytesseract
from PIL import Image
import os
import pandas as pd

# Set Tesseract path (update for your system)
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

input_dir = "right_half_screenshots"

# DataFrames to store results
correct_rating_df = pd.DataFrame(columns=['Text'])
remaining_rating_df = pd.DataFrame(columns=['Text'])

for filename in os.listdir(input_dir):
    if filename.endswith(".png"):
        print("===================================================================================")
        image_path = os.path.join(input_dir, filename)
        print(f"Processing: {image_path}")

        # Read and zoom image
        image = cv2.imread(image_path)
        scale_percent = 200  # Zoom level
        width = int(image.shape[1] * scale_percent / 100)
        height = int(image.shape[0] * scale_percent / 100)
        image = cv2.resize(image, (width, height), interpolation=cv2.INTER_LINEAR)

        # Convert to grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Apply adaptive thresholding for better OCR
        thresh = cv2.adaptiveThreshold(
            gray, 255,
            cv2.ADAPTIVE_THRESH_MEAN_C,
            cv2.THRESH_BINARY,
            blockSize=15,
            C=10
        )

        # Convert to PIL image
        pil_image = Image.fromarray(thresh)

        # OCR on the whole image
        custom_config = r'--oem 3 --psm 6 -c preserve_interword_spaces=1'
        full_text = pytesseract.image_to_string(pil_image, config=custom_config)
        # print(full_text)

        # Extract line-wise text
        lines = [line.strip() for line in full_text.split("\n") if line.strip()]
        
        start_printing = True
        for line in lines:
            if "PORTFOLIO" in line:
                start_printing = True

            if start_printing:
                print(line)
                print("-------------------------------------------------------------------------------------------------")
                correct_rating_df = pd.concat([correct_rating_df, pd.DataFrame([[line]], columns=['Text'])], ignore_index=True)

            if "Grand Total" in line and start_printing:
                break


Processing: right_half_screenshots\page_10_right_half.png
d Corporate Bonds (Excluding AA+ Rated Corporate Bonds).A relatively             Ame ae
-------------------------------------------------------------------------------------------------
tively high credit risk))
-------------------------------------------------------------------------------------------------
money market instruments across the yield curve & credit spectrum.There is no assurance that
-------------------------------------------------------------------------------------------------
t assure or guarantee any returns.
-------------------------------------------------------------------------------------------------
Instrument Type/ Issuer Name                                                                                  Rating             % of NAV
-------------------------------------------------------------------------------------------------
CorporateBond = ——<“—s——(<‘<—~™S™O™OOCOCOCOCOCOOOOOO 80,0296!
----------